# Notebook 04 — XGBoost
**Role:** Gradient boosting ML baseline  
**Data:** `Final_Cleaned_1m.parquet`  
**Features:** `total_traffic`, `avg_node_stress`, `replica_count`, `avg_response_time`  
**Target:** `total_cpu_demand`  
**Pipeline:** identical to LSTM — every-4th val split, log1p + MinMaxScaler  
**Metrics:** R², MAE, RMSE, MAPE → `results/all_models_results.csv`

In [ ]:
import random, os, warnings
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PATH    = str(PROJECT_ROOT / 'cleanData' / 'Final_Cleaned_1m.parquet')
PLOTS_DIR    = str(PROJECT_ROOT / 'plots')
RESULTS_CSV  = str(PROJECT_ROOT / 'results' / 'all_models_results.csv')
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(str(PROJECT_ROOT / 'results'), exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
                     'axes.grid':True,'grid.alpha':0.3,'font.size':11})

def save_result(model_name, r2, mae, rmse, mape):
    row = pd.DataFrame([{'model':model_name,'R2':round(float(r2),4),
        'MAE':round(float(mae),4),'RMSE':round(float(rmse),4),'MAPE':round(float(mape),2)}])
    if os.path.exists(RESULTS_CSV):
        ex = pd.read_csv(RESULTS_CSV)
        row = pd.concat([ex[ex['model']!=model_name], row], ignore_index=True)
    row.to_csv(RESULTS_CSV, index=False)
    print(f"  Saved \u2192 results/all_models_results.csv")

print(f"Project root : {PROJECT_ROOT}")
print(f"Data         : {DATA_PATH}")


Project root : D:\CEIPP\Cloud-ML\Cloud_Autoscale\Cloud_Autoscale
Data         : D:\CEIPP\Cloud-ML\Cloud_Autoscale\Cloud_Autoscale\cleanData\Final_Cleaned_1m.parquet


: 

## 1. Load Data

In [ ]:
print("Loading Cleaned 1-Minute Data...")
df_final_1m = pl.read_parquet(DATA_PATH)
print(f"Loaded Shape: {df_final_1m.shape}")
print(f"Columns: {df_final_1m.columns}")
print(f"Timestamp range: {df_final_1m['timestamp'].min()} to {df_final_1m['timestamp'].max()}")
print(f"Unique services: {df_final_1m['msname'].n_unique()}")


## 2. Build Sliding Windows 

In [ ]:
print("Building sliding windows...")

WINDOW_SIZE   = 5
PREDICT_STEPS = 1
FEATURES = ['total_traffic','avg_node_stress','replica_count','avg_response_time']

X_raw, y_raw = [], []
for name, group in df_final_1m.sort('timestamp').group_by('msname'):
    group   = group.with_columns(pl.col('avg_response_time').fill_null(0))
    X_array = group.select(FEATURES).to_numpy()
    y_array = group.select('total_cpu_demand').to_numpy()
    for i in range(WINDOW_SIZE, len(X_array) - PREDICT_STEPS + 1):
        X_raw.append(X_array[i-WINDOW_SIZE:i, :])
        y_raw.append(y_array[i:i+PREDICT_STEPS, 0])

X_raw = np.array(X_raw)
y_raw = np.array(y_raw).reshape(-1, PREDICT_STEPS)
print(f"X_raw Shape: {X_raw.shape}  ({WINDOW_SIZE} timesteps x {len(FEATURES)} features)")
print(f"y_raw Shape: {y_raw.shape}")
print(f"y_raw max:   {y_raw.max():.4f} cores | mean: {y_raw.mean():.4f} cores")


## 3. Split & Scale 

In [ ]:
print("Splitting into Train/Validation sets...")
val_indices   = np.arange(0, len(X_raw), 4)
train_indices = np.setdiff1d(np.arange(len(X_raw)), val_indices)

X_train_raw = X_raw[train_indices];  X_val_raw = X_raw[val_indices]
y_train_raw = y_raw[train_indices];  y_val_raw = y_raw[val_indices]
print(f"X_train: {X_train_raw.shape} | X_val: {X_val_raw.shape}")
print(f"y_train: {y_train_raw.shape} | y_val: {y_val_raw.shape}")

print("\nScaling with Log Transform...")
n_train, timesteps, n_features = X_train_raw.shape

y_train_log = np.log1p(y_train_raw);  y_val_log = np.log1p(y_val_raw)
X_train_log = np.log1p(X_train_raw);  X_val_log = np.log1p(X_val_raw)

X_scaler = MinMaxScaler()
X_train_scaled = X_scaler.fit_transform(
    X_train_log.reshape(-1, n_features)).reshape(n_train, timesteps, n_features)
X_val_scaled   = X_scaler.transform(
    X_val_log.reshape(-1, n_features)).reshape(X_val_log.shape)

y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train_log)
y_val_scaled   = y_scaler.transform(y_val_log)

# Flatten 3D -> 2D for sklearn, keep y as 1D
X_train_2d = X_train_scaled.reshape(n_train, -1)
X_val_2d   = X_val_scaled.reshape(len(X_val_scaled), -1)
y_train_1d = y_train_scaled.flatten()

print(f"\nX_train_2d: {X_train_2d.shape} | X_val_2d: {X_val_2d.shape}")
print(f"y_scaler range: {y_scaler.data_min_[0]:.4f} to {y_scaler.data_max_[0]:.4f} (log scale)")
print(f"Evaluated on val: {X_val_2d.shape[0]:,} samples")


## 4. Train XGBoost with Early Stopping

In [ ]:
from xgboost import XGBRegressor

val_cut = int(len(X_train_2d) * 0.9)
X_tr = X_train_2d[:val_cut]; y_tr = y_train_1d[:val_cut]
X_es = X_train_2d[val_cut:]; y_es = y_train_1d[val_cut:]

print("Training XGBoost with early stopping...")
xgb = XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=SEED,
    verbosity=0, early_stopping_rounds=20, eval_metric='rmse'
)
xgb.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
print(f"Best iteration: {xgb.best_iteration}")

pred_scaled = xgb.predict(X_val_2d)
# Inverse: unscale log -> expm1 -> CPU cores 
pred_actual = np.expm1(y_scaler.inverse_transform(pred_scaled.reshape(-1,1))).flatten()
true_actual = np.expm1(y_scaler.inverse_transform(y_val_scaled.reshape(-1,1))).flatten()

r2   = r2_score(true_actual, pred_actual)
mae  = mean_absolute_error(true_actual, pred_actual)
rmse = np.sqrt(mean_squared_error(true_actual, pred_actual))
mape = np.mean(np.abs((true_actual - pred_actual) / (true_actual + 1e-8))) * 100

print()
print('=' * 40)
print('MODEL EVALUATION REPORT')
print('=' * 40)
print(f'MAE  (Mean Absolute Error):     {mae:.4f} cores')
print(f'RMSE (Root Mean Squared Error): {rmse:.4f} cores')
print(f'R\u00b2   (Explained Variance):      {r2:.4f}')
print(f'MAPE (Mean Absolute % Error):   {mape:.2f}%')
print(f'Evaluated on:                   {len(true_actual):,} samples')
print('=' * 40)
save_result('XGBoost', r2, mae, rmse, mape)

## 5. Plots

In [ ]:
feat_names  = [f"{f}_t{j}" for j in range(timesteps) for f in FEATURES]
importances = pd.Series(xgb.feature_importances_, index=feat_names).sort_values()
evals = xgb.evals_result()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'XGBoost — Learning Curve, Importance & Prediction  (R²={r2:.4f})', fontsize=13, fontweight='bold')

axes[0].plot(evals['validation_0']['rmse'], color='#E24B4A', linewidth=1.0)
axes[0].axvline(xgb.best_iteration, color='#378ADD', linestyle='--', linewidth=1.2, label=f'Best={xgb.best_iteration}')
axes[0].set_title('Validation RMSE (log-scaled)'); axes[0].set_xlabel('Round'); axes[0].legend()

axes[1].barh(importances.index, importances.values, color='#D85A30', edgecolor='white', alpha=0.85)
axes[1].set_title('Feature importance'); axes[1].set_xlabel('Importance')

n_show = min(200, len(true_actual))
axes[2].plot(true_actual[:n_show], color='#378ADD', linewidth=1.0, label='Actual CPU cores')
axes[2].plot(pred_actual[:n_show], color='#D85A30', linewidth=1.0, linestyle='--', label='XGBoost predicted')
axes[2].set_title(f'First {n_show} validation samples'); axes[2].set_ylabel('CPU cores'); axes[2].legend()

for ax in axes: ax.grid(True, alpha=0.25)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, '04_xgb_results.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {out}")
print(f"Final — R²={r2:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f} | MAPE={mape:.2f}%")
